# 🫁 Respiratory Sound Analysis

This notebook analyzes respiratory sounds from Kaggle dataset using modern Python 3.9+ type hints:
- **Frequency Analysis** (FFT, spectral features)
- **Whistle Detection** (wheezes identification)  
- **Entropy-Complexity Analysis** (Bandt-Pompe permutation entropy)

Built with clean code architecture and full type safety.

---

## 📦 Install Dependencies

In [ ]:
!pip install -q kaggle numpy scipy matplotlib pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive connected!")
print("📁 kaggle.json will be stored at: /content/drive/MyDrive/kaggle.json")

---

## ⚙️ Setup Kaggle API and Download Dataset

In [ ]:
import os
import shutil
from google.colab import files

KAGGLE_JSON_DRIVE = '/content/drive/MyDrive/kaggle.json'
KAGGLE_JSON_LOCAL = os.path.expanduser('~/.kaggle/kaggle.json')

if os.path.exists(KAGGLE_JSON_DRIVE):
    print("✅ kaggle.json found in Google Drive!")
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy(KAGGLE_JSON_DRIVE, KAGGLE_JSON_LOCAL)
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
    print("✅ Kaggle API configured from Drive!")
else:
    print("📤 kaggle.json not found in Drive. Please upload it:")
    uploaded = files.upload()
    
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    
    with open(KAGGLE_JSON_LOCAL, 'wb') as f:
        f.write(list(uploaded.values())[0])
    
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
    
    shutil.copy(KAGGLE_JSON_LOCAL, KAGGLE_JSON_DRIVE)
    print("✅ Kaggle API configured and saved to Drive!")

In [ ]:
import os

LOCAL_DATASET_PATH = '/content/respiratory_sound_dataset'

if not os.path.exists(LOCAL_DATASET_PATH) or not os.listdir(LOCAL_DATASET_PATH):
    print("📥 Downloading dataset from Kaggle (3.69GB, ~5-10 min)...")
    !kaggle datasets download -d vbookshelf/respiratory-sound-database
    
    print("📦 Extracting...")
    os.makedirs(LOCAL_DATASET_PATH, exist_ok=True)
    !unzip -q respiratory-sound-database.zip -d {LOCAL_DATASET_PATH}
    !rm respiratory-sound-database.zip
    print("✅ Dataset downloaded and extracted!")
else:
    print("✅ Dataset already downloaded (delete folder manually to re-download)")

print(f"\n📊 Dataset ready: {LOCAL_DATASET_PATH}")
!ls -lh {LOCAL_DATASET_PATH} | head -20

---

## 📥 Clone Analysis Code Repository

In [ ]:
# Option 1: Clone only app/ folder (sparse checkout - faster)
!git clone --depth 1 --filter=blob:none --sparse https://github.com/incRED1bl/course_paper.git
%cd course_paper
!git sparse-checkout set app

print("✅ Cloned only app/ folder")

In [ ]:
import os
from scipy.io import wavfile

def load_respiratory_sounds(dataset_path, max_files=20):
    """Load audio files from the dataset."""
    signals = {}
    
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Path not found: {dataset_path}")
    
    audio_dir = None
    for root, dirs, files in os.walk(dataset_path):
        if 'audio_and_txt_files' in dirs:
            audio_dir = os.path.join(root, 'audio_and_txt_files')
            break
    
    if audio_dir and os.path.exists(audio_dir):
        print(f"📁 Found audio folder: {audio_dir}")
        wav_files = [f for f in os.listdir(audio_dir) if f.endswith('.wav')]
        print(f"📊 Total .wav files: {len(wav_files)}, loading: {min(max_files, len(wav_files))}")
        
        for wav_file in wav_files[:max_files]:
            file_path = os.path.join(audio_dir, wav_file)
            try:
                sample_rate, signal_data = wavfile.read(file_path)
                signals[wav_file] = {
                    'signal': signal_data,
                    'sample_rate': sample_rate
                }
            except Exception as e:
                print(f"⚠️ Failed to load {wav_file}: {e}")
    else:
        print(f"❌ audio_and_txt_files folder not found")
    
    return signals

print("✅ Load function ready!")

---

## 🔊 Load and Analyze Audio Files

In [ ]:
dataset_path = '/content/respiratory_sound_dataset'

print("📂 Loading audio files...")
signals = load_respiratory_sounds(dataset_path, max_files=20)

print(f"\n✅ Loaded {len(signals)} files")
print(f"📝 Examples: {list(signals.keys())[:3]}")

In [ ]:
sample_rate = list(signals.values())[0]['sample_rate']
print(f"📊 Sample rate: {sample_rate} Hz")
print(f"📈 Ready to extract features from {len(signals)} files")

### Create DataFrame with Features and Diagnosis

In [ ]:
import pandas as pd
from app.data_preprocessing import extract_features_batch

diagnosis_file = '/content/respiratory_sound_dataset/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv'
diagnosis_df = pd.read_csv(diagnosis_file)

print(f"📋 Available columns: {list(diagnosis_df.columns)}")
print(f"🔍 First rows of diagnosis file:")
display(diagnosis_df.head())

patient_col = diagnosis_df.columns[0]
diagnosis_col = diagnosis_df.columns[1]

rows = extract_features_batch(signals, embedding_dim=3, time_delay=1)

for row in rows:
    patient_id = row['patient_id']
    diagnosis = diagnosis_df[diagnosis_df[patient_col] == patient_id][diagnosis_col].values
    row['diagnosis'] = diagnosis[0] if len(diagnosis) > 0 else 'Unknown'

results_df = pd.DataFrame(rows)

print(f"\n✅ DataFrame created with {len(results_df)} samples")
print(f"\n📋 Diagnosis distribution:")
print(results_df['diagnosis'].value_counts())
print(f"\n🔍 First 5 rows:")
display(results_df.head())

---

## 📊 Visualization

### Time and Frequency Domain Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from app.features import extract_frequency_features

sample_files = list(signals.keys())[:3]

for filename in sample_files:
    signal_data = signals[filename]['signal']
    sample_rate_file = signals[filename]['sample_rate']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f'Analysis: {filename}', fontsize=14, fontweight='bold')
    
    ax1.plot(signal_data[:1000], linewidth=0.8, color='steelblue')
    ax1.set_title('Time Domain')
    ax1.set_xlabel('Samples')
    ax1.set_ylabel('Amplitude')
    ax1.grid(True, alpha=0.3)
    
    frequencies, magnitudes, _ = extract_frequency_features(signal_data, sample_rate_file)
    ax2.plot(frequencies, magnitudes, linewidth=1.2, color='steelblue')
    ax2.axvspan(400, 1600, alpha=0.2, color='red', label='Wheeze Range (400-1600 Hz)')
    ax2.axvspan(100, 400, alpha=0.1, color='green', label='Normal Breath (100-400 Hz)')
    ax2.set_title('Frequency Spectrum')
    ax2.set_xlabel('Frequency (Hz)')
    ax2.set_ylabel('Magnitude')
    ax2.set_xlim(0, 2000)
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='upper right', fontsize=9)
    
    plt.tight_layout()
    plt.show()

In [ ]:
sample_files = results_df['filename'].head(3).tolist()
energy_cols = ['low_freq_energy', 'mid_freq_energy', 'high_freq_energy']

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(sample_files))
width = 0.25

for i, col in enumerate(energy_cols):
    energies = results_df[results_df['filename'].isin(sample_files)][col].tolist()
    ax.bar(x + i*width, energies, width, label=col.replace('_', ' ').title(), alpha=0.8)

ax.set_title('Energy Distribution Across Frequency Bands', fontsize=14, fontweight='bold')
ax.set_xlabel('File')
ax.set_ylabel('Relative Energy')
ax.set_xticks(x + width)
ax.set_xticklabels([f[:15] + '...' for f in sample_files], rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Energy Distribution

In [ ]:
sample_data = results_df.head(10)

entropies = sample_data['entropy'].tolist()
complexities = sample_data['complexity'].tolist()
filenames = sample_data['filename'].tolist()

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(entropies, complexities, s=200, alpha=0.6, 
                    c=range(len(sample_data)), cmap='viridis',
                    edgecolors='black', linewidths=1.5)

for i in [0, len(sample_data)//2, -1]:
    ax.annotate(filenames[i][:10], (entropies[i], complexities[i]), 
                xytext=(5, 5), textcoords='offset points', 
                fontsize=9, alpha=0.7)

ax.set_title('Entropy-Complexity Plane', fontsize=14, fontweight='bold')
ax.set_xlabel('Normalized Entropy (H)', fontsize=12)
ax.set_ylabel('Statistical Complexity (C)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='File Index')

plt.tight_layout()
plt.show()

### Entropy-Complexity Plane

In [ ]:
print("💡 Add your model training code here")

---

## 🤖 Machine Learning Model Training

In [ ]:
import pickle

# Save the DataFrame with all features
with open('features.pkl', 'wb') as f:
    pickle.dump({
        'dataframe': results_df,
        'sample_rate': sample_rate
    }, f)

print("✅ Features saved to features.pkl")
print(f"   📊 Contains: DataFrame with {len(results_df)} samples")
print(f"   🔊 Sample rate: {sample_rate} Hz")

from google.colab import files
files.download('features.pkl')

---

## 💾 Save Results

In [ ]:
print("="*70)
print("  FINAL RESULTS")
print("="*70)
print(f"\n📊 Processed files: {len(signals)}")
print(f"🔬 DataFrame shape: {results_df.shape}")
print(f"📋 Features: {list(results_df.columns)}")
print(f"\n✅ Analysis complete!")
print(f"\n💾 Don't forget to download:")
print(f"   - features.pkl (DataFrame with all features)")
print(f"   - model.pkl (trained model - add ML code above)")
print(f"\n🚀 Upload files to VS Code for further work")